### 1. 使用function_call模式

In [10]:
# from pydantic_ai.common_tools import TavilySearchResult
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.tools import StructuredTool
from langchain_community.chat_models import ChatTongyi 
from langchain_classic.agents import AgentType,initialize_agent
from langchain_core.tools import Tool
import os
import dotenv
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_agent
from langchain_classic.agents import AgentExecutor
from langchain_classic.agents import create_tool_calling_agent

In [17]:
from langgraph.prebuilt import create_react_agent


dotenv.load_dotenv()
#配置环境变量
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["DASHSCOPE_API_KEY"] = os.getenv("DASHSCOPE_API_KEY")
#获取工具实例
Search = TavilySearchResults(
    max_result=3,

)

#封装工具
search_tool = Tool(
    name="Search",
    func=Search.run,
    description="一个天气搜索工具，输入一个字符串，返回搜索结果的字符串",
)

#获取模型
llm = ChatTongyi(
    temperature=0.8,
    model="qwen-flash",
)

#提供提示词模板
prompt_template = ChatPromptTemplate.from_messages([
    ("system","你是一个乐于主人的ai助手，根据用户提问，必要时调用Search工具，使用互联网检索数据"),
    ("human","{input}"),
    ("placeholder","{agent_scratchpad}")
])

#获取agent实例
agent = create_tool_calling_agent(
    llm=llm,
    prompt=prompt_template,
    tools=[search_tool],
)

#获取AgentExecutor实例
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool],
    verbose=True
)

#调用agent
result = agent_executor.invoke({"input":"今天北京天气怎么样？给我日期＋天气"})

#打印结果
print(result)



> Entering new AgentExecutor chain...

Invoking: `Search` with `今天北京天气`


[{'title': '北京 - 百度', 'url': 'https://weathernew.pae.baidu.com/weathernew/pc?query=%E5%8C%97%E4%BA%AC%E5%A4%A9%E6%B0%94&srcid=4982', 'content': 'Image 1\n\n北京 今天：晴  8°~22°C  东南风3级\n\n04月11日 周六 农历二月廿四\n\n16°\n\n38 优\n\n晴 东南风  4级 \n\n湿度50%日出05:43 日落18:48\n\n24小时预报\n\n    温度 \n    降水量 \n    风力风向 \n    空气质量 \n\n22°C 15°C 8°C\n\n现在\n\n10:00\n\n11:00\n\n12:00\n\n13:00\n\n14:00\n\n15:00\n\n16:00\n\n17:00\n\n18:00\n\n19:00\n\n20:00\n\n21:00\n\n22:00\n\n23:00\n\n00:00\n\n01:00\n\n02:00\n\n03:00\n\n04:00\n\n05:00\n\n06:00\n\n07:00\n\n08:00\n\n15天天气预报\n\n   昨天\n\n04/10\n\n3级\n\n优\n\n   今天\n\n04/11\n\n3级\n\n优\n\n   明天\n\n04/12\n\n1级\n\n轻度\n\n   周一\n\n04/13\n\n2级\n\n良\n\n   周二\n\n04/14\n\n3级\n\n良\n\n   周三\n\n04/15\n\n2级\n\n轻度\n\n   周四\n\n04/16\n\n2级\n\n轻度\n\n   周五\n\n04/17\n\n2级\n\n良\n\n   周六\n\n04/18\n\n3级\n\n   周日\n\n04/19\n\n3级\n\n   周一\n\n04/20\n\n3级\n\n   周二\n\n04/21\n\n2级\n\n   周三\n\n04/22\n\n2级\n\n   周四\n\n04/23\n\n3

### 2. 使用react模式

2.1.使用PromptTemplate

In [20]:

from langchain_classic.agents import create_react_agent
from langchain_core.prompts import PromptTemplate

dotenv.load_dotenv()
#配置环境变量
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["DASHSCOPE_API_KEY"] = os.getenv("DASHSCOPE_API_KEY")
#获取工具实例
Search = TavilySearchResults(
    max_result=3,

)

#封装工具
search_tool = Tool(
    name="Search",
    func=Search.run,
    description="一个天气搜索工具，输入一个字符串，返回搜索结果的字符串",
)

#获取模型
llm = ChatTongyi(
    temperature=0.8,
    model="qwen-flash",
)

#提供提示词模板
template ="""
Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}
"""
prompt_template = PromptTemplate.from_template(
    template=template,
)

#获取agent实例
agent = create_react_agent(
    llm=llm,
    prompt=prompt_template,
    tools=[search_tool],
)

#获取AgentExecutor实例
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool],
    verbose=True
)

#调用agent
result = agent_executor.invoke({"input":"今天北京天气怎么样？给我日期＋天气"})

#打印结果
print(result)



> Entering new AgentExecutor chain...
Thought: 我需要查询今天北京的天气情况，包括日期和天气状况。
Action: Search
Action Input: 今天北京天气怎么样[{'title': '北京 - 百度', 'url': 'https://weathernew.pae.baidu.com/weathernew/pc?query=%E5%8C%97%E4%BA%AC%E5%A4%A9%E6%B0%94&srcid=4982', 'content': 'Image 1\n\n北京 今天：晴  8°~22°C  东南风3级\n\n04月11日 周六 农历二月廿四\n\n16°\n\n38 优\n\n晴 东南风  4级 \n\n湿度50%日出05:43 日落18:48\n\n24小时预报\n\n    温度 \n    降水量 \n    风力风向 \n    空气质量 \n\n22°C 15°C 8°C\n\n现在\n\n10:00\n\n11:00\n\n12:00\n\n13:00\n\n14:00\n\n15:00\n\n16:00\n\n17:00\n\n18:00\n\n19:00\n\n20:00\n\n21:00\n\n22:00\n\n23:00\n\n00:00\n\n01:00\n\n02:00\n\n03:00\n\n04:00\n\n05:00\n\n06:00\n\n07:00\n\n08:00\n\n15天天气预报\n\n   昨天\n\n04/10\n\n3级\n\n优\n\n   今天\n\n04/11\n\n3级\n\n优\n\n   明天\n\n04/12\n\n1级\n\n轻度\n\n   周一\n\n04/13\n\n2级\n\n良\n\n   周二\n\n04/14\n\n3级\n\n良\n\n   周三\n\n04/15\n\n2级\n\n轻度\n\n   周四\n\n04/16\n\n2级\n\n轻度\n\n   周五\n\n04/17\n\n2级\n\n良\n\n   周六\n\n04/18\n\n3级\n\n   周日\n\n04/19\n\n3级\n\n   周一\n\n04/20\n\n3级\n\n   周二\n\n04/21\n\n2级\n\n   周三\

2.2.使用ChatPromptTemplate

In [28]:

from langchain_classic.agents import create_react_agent

dotenv.load_dotenv()
#配置环境变量
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["DASHSCOPE_API_KEY"] = os.getenv("DASHSCOPE_API_KEY")
#获取工具实例
Search = TavilySearchResults(
    max_result=3,

)

#封装工具
search_tool = Tool(
    name="Search",
    func=Search.run,
    description="一个天气搜索工具，输入一个字符串，返回搜索结果的字符串",
)

#获取模型
llm = ChatTongyi(
    temperature=0.8,
    model="qwen-flash",
)

#提供提示词模板
system_template = """
Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!


"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system",system_template),
    ("human","{input}"),
    ("system","{agent_scratchpad}"),
])


#获取agent实例
agent = create_react_agent(
    llm=llm,
    prompt=prompt_template,
    tools=[search_tool],
)

#获取AgentExecutor实例
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool],
    verbose=True
)

#调用agent
result = agent_executor.invoke({"input":"今天北京天气怎么样？给我日期＋天气"})

#打印结果
print(result)



> Entering new AgentExecutor chain...
Thought: 我需要查询今天北京的天气情况，包括日期和天气状况。
Action: Search
Action Input: 今天北京天气怎么样[{'title': '北京 - 天气预报', 'url': 'https://weather.cma.cn/web/text/HB/ABJ.html', 'content': '| 朝阳 | 多云 | 南风 | 微风 | 16℃ | 晴 | 南风 | 微风 | 4℃ | 详情>> |\n| 海淀 | 多云 | 南风 | 微风 | 16℃ | 晴 | 南风 | 微风 | 3℃ | 详情>> |\n| 石景山 | 多云 | 南风 | 微风 | 16℃ | 晴 | 南风 | 微风 | 3℃ | 详情>> | [...] | 朝阳 | 晴 | 东南风 | 微风 | 14℃ | 晴 | 东南风 | 微风 | 2℃ | 详情>> |\n| 海淀 | 晴 | 南风 | 微风 | 16℃ | 晴 | 东南风 | 微风 | 1℃ | 详情>> |\n| 石景山 | 晴 | 南风 | 微风 | 15℃ | 晴 | 北风 | 微风 | 0℃ | 详情>> | [...] | 朝阳 | 晴 | 北风 | 微风 | 19℃ | 晴 | 北风 | 微风 | 8℃ | 详情>> |\n| 海淀 | 晴 | 东北风 | 微风 | 21℃ | 晴 | 北风 | 微风 | 8℃ | 详情>> |\n| 石景山 | 晴 | 东北风 | 微风 | 20℃ | 晴 | 北风 | 微风 | 8℃ | 详情>> |', 'score': 0.9969957}, {'title': '北京 - 百度', 'url': 'https://weathernew.pae.baidu.com/weathernew/pc?query=%E5%8C%97%E4%BA%AC%E5%A4%A9%E6%B0%94&srcid=4982', 'content': 'Image 1\n\n北京 今天：晴  8°~22°C  东南风3级\n\n04月11日 周六 农历二月廿四\n\n16°\n\n38 优\n\n晴 东南风  4级 \n\n湿度50%日出05:43 日落18:48\n\n24小时预报\n\n   